# 05_prepare_data_for_models

#### Import libraries

In [1]:
import pandas as pd
import numpy as np

#### Import `df_exact_with_dhw_and_max` database

In [ ]:
# Go up one level from notebooks/ to 01_data_assembly/
root = Path("..")
file_path_import = root / "data" / "intermediate" / "df_exact_with_dhw_and_max.xlsx"
df = pd.read_excel(file_path_import)

#### Data counts and types

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21008 entries, 0 to 21007
Data columns (total 58 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   FID                           21008 non-null  int64         
 1   OCEAN_REGION                  20965 non-null  object        
 2   COUNTRY                       21008 non-null  object        
 3   LOCATION                      20225 non-null  object        
 4   SITE_NAME                     19276 non-null  object        
 5   LATITUDE                      21008 non-null  object        
 6   LONGITUDE                     21008 non-null  float64       
 7   DAY                           21008 non-null  int64         
 8   MONTH                         21008 non-null  int64         
 9   YEAR                          21008 non-null  int64         
 10  MONTHS_SINCE_PEAK             0 non-null      float64       
 11  MAX_DEPTH_m                 

#### Clean Latitude/Longitude Columns
Clean hidden formatting issues to prevent conversion errors and ensure the coordinates can be used reliably for mapping and analysis.

In [5]:
def clean_numeric(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace("\xa0", "", regex=False)
        .str.replace("−", "-", regex=False)
    )

df["LATITUDE"] = pd.to_numeric(clean_numeric(df["LATITUDE"]), errors="coerce")
df["LONGITUDE"] = pd.to_numeric(clean_numeric(df["LONGITUDE"]), errors="coerce")

#### Re-checking

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21008 entries, 0 to 21007
Data columns (total 58 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   FID                           21008 non-null  int64         
 1   OCEAN_REGION                  20965 non-null  object        
 2   COUNTRY                       21008 non-null  object        
 3   LOCATION                      20225 non-null  object        
 4   SITE_NAME                     19276 non-null  object        
 5   LATITUDE                      21008 non-null  float64       
 6   LONGITUDE                     21008 non-null  float64       
 7   DAY                           21008 non-null  int64         
 8   MONTH                         21008 non-null  int64         
 9   YEAR                          21008 non-null  int64         
 10  MONTHS_SINCE_PEAK             0 non-null      float64       
 11  MAX_DEPTH_m                 

#### Create Categorical Variables and Spatial Clusters
This block creates categorical variables and assigns geographic observations into custom spatial clusters based on country, longitude, latitude, and ocean region.
Spatial clustering helps organize observations into ecologically meaningful regions for modelling purposes

In [8]:
# YEAR is a factor equivalent
df["YEAR"] = df["YEAR"].astype("category")

# Build spatial_cluster
conditions = [
    (df["COUNTRY"].eq("Australia") & df["LONGITUDE"].gt(143)),
    (df["COUNTRY"].eq("Australia") & df["LONGITUDE"].le(143)),

    (df["COUNTRY"].eq("India") & df["LONGITUDE"].gt(90)),
    (df["COUNTRY"].eq("India") & df["LONGITUDE"].le(90)),

    (
        df["COUNTRY"].eq("France (Indian Ocean)")
        & df["LATITUDE"].between(-21.7, -20.5, inclusive="both")
        & df["LONGITUDE"].between(55, 56, inclusive="both")
    ),
    (
        df["COUNTRY"].eq("France (Indian Ocean)")
        & df["LATITUDE"].between(-13.08, -12.5, inclusive="both")
        & df["LONGITUDE"].between(44.9, 46.1, inclusive="both")
    ),
    (
        df["COUNTRY"].eq("France (Indian Ocean)")
        & df["LATITUDE"].between(-11.6, -11.56, inclusive="both")
        & df["LONGITUDE"].between(47.28, 47.38, inclusive="both")
    ),
    (
        df["COUNTRY"].eq("France (Indian Ocean)")
        & df["LATITUDE"].between(-23, -17, inclusive="both")
        & df["LONGITUDE"].between(39, 43, inclusive="both")
    ),

    (df["COUNTRY"].eq("United States") & df["OCEAN_REGION"].eq("Caribbean")),
    (df["COUNTRY"].eq("United States") & df["OCEAN_REGION"].eq("Pacific Ocean")),
]

choices = [
    "Australia_GBR",
    "Australia_nonGBR",
    "India_Andaman",
    "India_main",
    "France_Reunion",
    "France_Mayotte",
    "France_Glorieuses",
    "France_Eparse",
    "US_continent",
    "US_Hawaii",
]

df["spatial_cluster"] = np.select(
    conditions,
    choices,
    default=df["COUNTRY"]
)

# spatial_cluster as factor equivalent
df["spatial_cluster"] = df["spatial_cluster"].astype("category")

#### `recode_severity` function
Recode and order Severity categories. This block converts numeric severity codes into ordered categorical labels for modelling purposes.

In [9]:
severity_dtype = pd.CategoricalDtype(
    categories=["None", "Mild", "Moderate", "Severe"],
    ordered=True
)

severity_map = {
    0: "None",
    1: "Mild",
    2: "Moderate",
    3: "Severe"
}

def recode_severity(data):
    data = data.copy()
    data["SEVERITY_CODE"] = (
        data["SEVERITY_CODE"]
        .map(severity_map)
        .astype(severity_dtype)
    )
    return data

#### `df_pre` training dataset
Filter and prepare pre-peak Severity data for statistical modeling. This block filters the dataset to retain valid observations collected before or at the peak DHW event and then recodes severity values into ordered categories.
Removes inconsistent observations and exclude records where:
- `DHW == 0`
- but bleaching severity is recorded as:
  - Mild
  - Moderate
  - Severe

These combinations may indicate:
- data entry issues
- bleaching potentially driven by non-thermal stressors

In [10]:
df_pre = df.loc[
    (df["DAYS_FROM_MAX_DHW"] <= 0)
    & df["SEVERITY_CODE"].notna()
    & (df["SEVERITY_CODE"] != -1)
    & df["DHW"].notna()
    & df["spatial_cluster"].notna()
    & df["YEAR"].notna()
    & ~((df["DHW"] == 0) & (df["SEVERITY_CODE"].isin([1, 2, 3])))
].copy()

df_pre = recode_severity(df_pre)

#### `df_pre_screening` dataset
Filter and prepare pre-peak screening dataset for the post-modeling phase. 
This step creates a refined dataset for the post-modeling phase of the project. It excludes observations with negligible thermal stress exposure, focusing the analysis on locations experiencing at least some level of heat stress and retaining only biologically relevant bleaching observations. When multiple records occur at the same location during the same heat stress event, only the report closest to the peak thermal stress is retained. This helps reduce duplicate observations and ensures consistency in the screening dataset.

In [11]:
df_pre_screening = df.loc[
    (df["DAYS_FROM_MAX_DHW"] <= 0)
    & df["SEVERITY_CODE"].notna()
    & (df["SEVERITY_CODE"] != -1)
    & df["DHW"].notna()
    & df["MAX_ANNUAL_DHW"].notna()
    & df["spatial_cluster"].notna()
    & df["YEAR"].notna()
    & (df["DHW"] >= 0.5)
    & (df["IS_BEST_REPORT"] == True)
].copy()

df_pre_screening = recode_severity(df_pre_screening)

#### `df_post` training dataset
Filter and prepare post-peak Severity data for statistical modeling. This block filters the dataset to retain valid observations collected after or at the peak DHW event and then recodes severity values into ordered categories.
Removes inconsistent observations and exclude records where:
- `DHW == 0`
- but bleaching severity is recorded as:
  - Mild
  - Moderate
  - Severe

These combinations may indicate:
- data entry issues
- bleaching potentially driven by non-thermal stressors

In [12]:
df_post = df.loc[
    (df["DAYS_FROM_MAX_DHW"] >= 0)
    & df["SEVERITY_CODE"].notna()
    & (df["SEVERITY_CODE"] != -1)
    & df["MAX_ANNUAL_DHW"].notna()
    & df["DAYS_FROM_MAX_DHW"].notna()
    & df["spatial_cluster"].notna()
    & df["YEAR"].notna()
    & ~((df["DHW"] == 0) & (df["SEVERITY_CODE"].isin([1, 2, 3])))
].copy()

df_post = df_post.drop(df_post.index[[6355, 9841]])

df_post = recode_severity(df_post)

#### `df_post_screening` dataset
Filter and prepare the post-peak screening dataset for the post-modeling phase. 
This step creates a refined dataset for the post-modeling phase of the project. It excludes observations with negligible thermal stress exposure, focusing the analysis on locations experiencing at least some level of heat stress and retaining only biologically relevant bleaching observations. When multiple records occur at the same location during the same heat stress event, only the report closest to the peak thermal stress is retained. This helps reduce duplicate observations and ensures consistency in the screening dataset.

In [13]:
df_post_screening = df.loc[
    (df["DAYS_FROM_MAX_DHW"] >= 0)
    & df["SEVERITY_CODE"].notna()
    & (df["SEVERITY_CODE"] != -1)
    & df["MAX_ANNUAL_DHW"].notna()
    & df["DAYS_FROM_MAX_DHW"].notna()
    & df["spatial_cluster"].notna()
    & df["YEAR"].notna()
    & ~((df["DHW"] == 0) & (df["SEVERITY_CODE"].isin([1, 2, 3])))
].copy()

df_post_screening = df_post_screening.drop(df_post_screening.index[[6355, 9841]])

df_post_screening = df_post_screening.loc[
    (df_post_screening["DHW"] >= 0.5)
    & (df_post_screening["IS_BEST_REPORT"] == True)
].copy()

df_post_screening = recode_severity(df_post_screening)

#### Export datasets

In [ ]:
df_pre.to_csv("../data/final/df_pre.csv", index=False)
df_pre_screening.to_csv("../data/final/df_pre_screening.csv", index=False)
df_post.to_csv("../data/final/df_post.csv", index=False)
df_post_screening.to_csv("../data/final/df_post_screening.csv", index=False)